# silver layer

### 1-taxi_trip

In [ ]:
import dlt
from pyspark.sql import functions as F

silver_expectations = {
    "valid_trip_distance": "trip_distance > 0",
    "valid_passenger_count": "passenger_count IS NOT NULL AND passenger_count > 0",
    "valid_VendorID": "VendorID IS NOT NULL AND VendorID IN (1, 2)" ,
    "valid_pickup_time": "tpep_pickup_datetime IS NOT NULL" ,
    "valid_dropoff_time": "tpep_dropoff_datetime IS NOT NULL" ,
    "valid_pick_LocationID": "Pickup_Location_ID IS NOT NULL AND Pickup_Location_ID BETWEEN 1 AND 265" ,
    "valid_drop_LocationID":  "dropoff_Location_ID IS NOT NULL AND dropoff_Location_ID BETWEEN 1 AND 265",
    "valid_fare_amount": "fare_amount > 0" ,
    "valid_mta_tax": "mta_tax IS NOT NULL AND mta_tax IN (0.50, 0,00)" ,
    "valid_tip_amount": "tip_amount >= 0" ,
    "valid_tolls_amount": "tolls_amount >= 0",
    "valid_improvement_surcharge": "improvement_surcharge >= 0" ,
    "valid_congestion_surcharge": "congestion_surcharge >= 0" ,
    "valid_Airport_fee": "Airport_fee >= 0" ,
    "valid_cbd_congestion_fee": "cbd_congestion_fee >= 0" ,
    "valid_total_amount": "total_amount IS NOT NULL AND total_amount > 0 AND total_amount <= 1000" ,
    "valid_total_amount_calculation": """
        ABS(
            total_amount - (
                fare_amount + 
                extra + 
                mta_tax + 
                tip_amount + 
                tolls_amount + 
                improvement_surcharge + 
                congestion_surcharge + 
                airport_fee + 
                cbd_congestion_fee
            )
        ) <= 0.01
    """

}

@dlt.table(
    name="silver_trips",
    comment="Cleaned and transformed taxi trip data for silver layer",
   
)
@dlt.expect_all_or_drop(silver_expectations)
def silver_trip():
    return(
        dlt.read_stream("bronze_taxi_trip")
        .withColumn("VendorID", F.col("VendorID"))
        .withColumn("pickup_time", F.col("tpep_pickup_datetime").cast("timestamp"))
        .withColumn("dropoff_time", F.col("tpep_dropoff_datetime").cast("timestamp"))
        .withColumn(
            "passenger_count",
          
            F.when((F.col("passenger_count") >= 1) & (F.col("passenger_count") <= 6), F.col("passenger_count"))
             .otherwise(None)
        )

        .filter( (F.col("trip_distance") > 0) &  (F.col("trip_distance") <= 150)) 
        
        .withColumn(
            "Rate_code_ID",
          
            F.when((F.col("RatecodeID") >= 1) & (F.col("RatecodeID") <= 6), F.col("RatecodeID"))
             .otherwise(None)
        )
        .withColumn("Pickup_Location_ID", F.col("PULocationID"))
        .withColumn("dropoff_Location_ID", F.col("DOLocationID"))

         
        .withColumn(
            "payment_type",
            F.when((F.col("payment_type") >= 1) & (F.col("payment_type") <= 6), F.col("payment_type"))
             .otherwise(5)
        )  

        .filter((F.col("fare_amount") > 0) & (F.col("fare_amount") <= 500))
        .filter((F.col("extra") >= 0) & (F.col("extra") <= 20))
        .withColumn("mta_tax", F.col("mta_tax"))
        .withColumn("tips_amount", F.col("tip_amount"))

        .select(
        "VendorID",
        "pickup_time",
        "dropoff_time",
        "passenger_count",
        "trip_distance",
        "Rate_code_ID",
        
        "Pickup_Location_ID",
        "dropoff_Location_ID",
        
        "payment_type",
        "fare_amount",
        "extra",
        "mta_tax",
        "tips_amount",
        "tolls_amount",
        "improvement_surcharge",
        "congestion_surcharge",
        "airport_fee",
        "cbd_congestion_fee",
        "total_amount"
    )
        

    )    
         
        


 ### 2-taxi_zone

In [ ]:
import dlt
from pyspark.sql import functions as F


silver_zone_expectations = {
   
    "valid_location_id": "location_id IS NOT NULL AND location_id > 0",
   
    "valid_zone_name": "zone IS NOT NULL AND zone != ''"
}

@dlt.table(
    name="silver_taxi_zones",
    comment="Cleaned taxi zone lookup table for silver layer"
)
@dlt.expect_all_or_drop(silver_zone_expectations)
def silver_taxi_zones():
    return (
        
        dlt.read_stream("bronze_taxi_zone")  
        
       
        .withColumn("location_id", F.col("LocationID").cast("integer"))

         .withColumn(  "borough", 
            F.when(F.col("borough") == "Unknown", "N/A")
            .otherwise(F.col("borough"))
        )
         
        .withColumn("zone", F.col("Zone"))
        .withColumn("service_zone", F.col("service_zone"))
        
   
        .select(
            "location_id",
            "borough",
            "zone",
            "service_zone"
        )
    )